Polished Execution Flow

This notebook orchestrates the following:

- Load persona
- Simulate user context
- Run multi-agent debate
- Run counterfactual reasoning
- Retrieve cross-domain matches
- Generate recommendations
- Store conversational memory
- Explain recommendations

In [2]:

import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
import pandas as pd

In [4]:
# LOAD PERSONA DATASET

persona_df = pd.read_csv(
    "../outputs/persona_dataset.csv"
)

persona_df.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster,archetype
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444,0.138507,0.222461,2,Harsh Critic
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273,0.215294,0.162739,3,Emotional Storyteller
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667,0.298114,0.206382,3,Emotional Storyteller
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667,0.425713,0.245910,0,Warm Optimist
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455,0.255776,0.355440,1,Reactive Reviewer


In [5]:
lagos_df = pd.read_csv(
    "../data/external/clean_lagos_restaurants.csv"
)

lagos_df.head()

,Unnamed: 0,author_name,review_title,review_text,author_rating,visit_date,overall_rating,restaurant_name
0,0,N9599MZaisham,Less than basic taste,For a brand that claims to have one of the bes...,2.0,01/09/2022,2.0,01 Shawarma
1,1,T-Africa2000,Much improved,We had a business dinner at 1415 this week and...,4.0,01/01/2020,4.0,1415 Steakhouse Seafood Restaurant
2,2,sunilt1960,Calm and Relaxing,Had dinner at this restaurant while staying in...,4.5,01/02/2019,4.0,1415 Steakhouse Seafood Restaurant
3,3,ooobabatunde,Better,Thank you for staying with us in Eko Hotels & ...,4.5,01/01/2019,4.0,1415 Steakhouse Seafood Restaurant
4,4,MrTraveller420,1415 Steakhouse,Went over to the old location of the Steakhous...,4.0,01/11/2018,4.0,1415 Steakhouse Seafood Restaurant


In [6]:
unified_behavior_df = pd.read_csv(
    "../outputs/unified_behavior_dataset.csv"
)

unified_behavior_df.head()

,domain,user_id,item_id,review_text,rating,timestamp
0,yelp,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,"If you decide to eat here, just be aware it is...",3.0,2018-07-07 22:09:11
1,yelp,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,I've taken a lot of spin classes over the year...,5.0,2012-01-03 15:28:18
2,yelp,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,Family diner. Had the buffet. Eclectic assortm...,3.0,2014-02-05 20:30:30
3,yelp,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,"Wow! Yummy, different, delicious. Our favo...",5.0,2015-01-04 00:01:03
4,yelp,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,Cute interior and owner (?) gave us tour of up...,4.0,2017-01-14 20:54:15


In [9]:
import pandas as pd

def print_schema(filepath, name):
    print(f"\n{'='*60}")
    print(f"Schema for: {name}")
    print(f"File: {filepath}")
    print(f"{'='*60}")
    df = pd.read_csv(filepath)
    print(f"Shape: {df.shape}")
    print(f"\nColumns:\n{df.columns.tolist()}")
    print(f"\nData types:\n{df.dtypes}")
    print(f"\nFirst 2 rows:\n{df.head(2).to_string()}")
    print(f"\nMissing values per column:\n{df.isnull().sum()}")

# Adjust paths to match your actual file locations
print_schema("../outputs/persona_dataset.csv", "Persona Dataset")
print_schema("../data/external/clean_lagos_restaurants.csv", "Lagos Restaurants")
print_schema("../outputs/unified_behavior_dataset.csv", "Unified Behavior Dataset")


Schema for: Persona Dataset
File: ../outputs/persona_dataset.csv
Shape: (300, 9)

Columns:
['user_id', 'avg_rating', 'rating_variance', 'review_count', 'avg_review_length', 'avg_sentiment', 'sentiment_variance', 'cluster', 'archetype']

Data types:
user_id                   str
avg_rating            float64
rating_variance       float64
review_count            int64
avg_review_length     float64
avg_sentiment         float64
sentiment_variance    float64
cluster                 int64
archetype                 str
dtype: object

First 2 rows:
                  user_id  avg_rating  rating_variance  review_count  avg_review_length  avg_sentiment  sentiment_variance  cluster              archetype
0  -EX1hrPRBqNkVavtMllTCA    3.250000         1.441725            36         390.194444       0.138507            0.222461        2           Harsh Critic
1  -M7fUg7FrdGctKr5f_eMUQ    4.090909         1.341963            22         358.727273       0.215294            0.162739        3  Emotiona

In [12]:
persona_df = pd.read_csv("../outputs/persona_dataset.csv")
unified_df = pd.read_csv("../outputs/unified_behavior_dataset.csv")

# Merge on user_id (left join: keep all reviews, add archetype & other persona columns)
unified_df = unified_df.merge(
    persona_df[['user_id', 'archetype', 'cluster', 'avg_rating', 'dominant_value']],
    on='user_id',
    how='left'
)

# Save as new file
unified_df.to_csv("../outputs/unified_behavior_with_archetype.csv", index=False)

In [13]:
# Load files
persona_df = pd.read_csv("../outputs/persona_dataset.csv")
unified_df = pd.read_csv("../outputs/unified_behavior_dataset.csv")

# Merge on user_id (left join keeps all reviews, adds archetype where available)
unified_df = unified_df.merge(
    persona_df[['user_id', 'archetype', 'dominant_value', 'avg_rating']],
    on='user_id',
    how='left'
)

# Save back (overwrite or create a new file)
unified_df.to_csv("../outputs/unified_behavior_dataset_with_archetype.csv", index=False)

In [ ]:
from src.recommender.agentic_orchestrator import (
    run_agentic_recommendation_pipeline
)

In [ ]:
persona = persona_df.sample(
    1
).iloc[0]

business = lagos_df.sample(
    1
).iloc[0]

In [ ]:
output = run_agentic_recommendation_pipeline(

    persona_row=persona,

    business_row=business,

    unified_behavior_df=unified_behavior_df,

    context_name="celebration",

    user_id="user_001"
)

In [ ]:
output

{'memory_context': "\n    User previously interacted with:\n    ['goodreads', 'goodreads', 'goodreads', 'goodreads', 'goodreads']\n    ",
 'debate_output': {'final_decision': 'recommend',
  'agent_discussions': [{'agent': 'Emotional Agent',
    'decision': 'neutral',
    'reason': 'Emotion was not a major factor.'},
   {'agent': 'Value Agent',
    'decision': 'positive',
    'reason': 'Pricing aligns with expectations.'},
   {'agent': 'Analytical Agent',
    'decision': 'neutral',
    'reason': 'Moderate reputation detected.'},
   {'agent': 'Social Agent',
    'decision': 'positive',
    'reason': 'Context supports social engagement.'}]},
 'counterfactuals': [],
 'recommendations': [{'domain': 'goodreads',
   'rating': 4.0,
   'recommendation_preview': 'Boy in a Band is a poignant, emotional and very sexy story of Morgan and Matthew, who meet as 12 year olds and in the Seventies. This is an extremely nostalgic coming of age story ',
   'explanation': 'This recommendation was selected b